# Assignment 04 - Notebook 02: Mạng nơ-ron tích chập 1D bằng PyTorch trên dữ liệu Diabetes

**Học phần:** Phát triển các Hệ thống Thông minh
**Sinh viên:** Nguyễn Duy Nghĩa · **Mã sinh viên:** B23DCCN600 · **Lớp:** D23CTPM01
**Giảng viên:** PGS.TS Trần Đình Quế
**Học kỳ:** Học kỳ 1 năm học 2026 - 2027

## Mục tiêu của notebook

Notebook này hiện thực **đúng kiến trúc 1D CNN của Notebook 01** nhưng bằng **PyTorch 2.9.1**,
trên **cùng một phép chia dữ liệu, cùng seed, cùng siêu tham số**. Mục đích không phải đạt điểm số
cao hơn, mà là tạo ra một phép so sánh có kiểm soát giữa hai cách hiện thực: một bên tự viết toàn
bộ lan truyền ngược bằng NumPy, một bên dùng bộ vi phân tự động `autograd` của framework.

Ba câu hỏi notebook này trả lời:

1. Cài đặt NumPy thủ công ở Notebook 01 có **tương đương về mặt số học** với `nn.Conv1d` chuẩn của
   PyTorch hay không? Mục 5 sẽ đối chứng trực tiếp đầu ra của hai cài đặt trên cùng một bộ trọng số.
2. Việc chuyển sang framework **thay đổi kết quả tới mức nào**, và phần chênh lệch còn lại đến từ
   đâu?
3. Framework **rút gọn được bao nhiêu công sức**, và cái giá phải trả về mặt hiểu biết là gì?

## Điểm khác biệt về mặt khái niệm so với Notebook 01

Ở Notebook 01, mỗi tầng phải tự cài `backward` bằng tay và mỗi công thức đạo hàm là một dòng code
có thể sai. Ở đây, PyTorch xây **đồ thị tính toán động** trong lúc chạy lượt thuận: mỗi phép toán
trên tensor có `requires_grad=True` đều ghi lại một nút cùng hàm đạo hàm tương ứng. Khi gọi
`loss.backward()`, thư viện duyệt ngược đồ thị theo thứ tự tô-pô và áp quy tắc chuỗi tự động.

Nói cách khác, toàn bộ phần khó nhất của Notebook 01, thứ mà chúng tôi phải kiểm định bằng sai
phân hữu hạn, ở đây được thay bằng **một lời gọi hàm duy nhất**. Đó là lý do báo cáo thực hiện
Notebook 01 trước: chỉ sau khi đã tự tay dẫn và kiểm chứng từng công thức đạo hàm thì `backward()`
mới thôi là một hộp đen.

Cần lưu ý một quy ước bố cục tensor: PyTorch dùng **kênh trước** (channels-first), tức
`(N, C, L)`, đúng bằng bố cục mà Notebook 01 đã chọn. TensorFlow ở Notebook 03 dùng **kênh sau**
`(N, L, C)`, nên sẽ cần chuyển vị. Đây là một nguồn lỗi phổ biến khi chuyển mô hình giữa hai
framework và chúng tôi nêu rõ ngay từ đầu.

## 1. Nhập thư viện và cố định seed

In [1]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["font.sans-serif"] = ["Segoe UI", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"

DATA_PATH = "../data/diabetes_prediction_dataset.csv"
FIG_DIR, REP_DIR = "../reports/figures", "../reports"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch      :", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())
print("Thiết bị     :", DEVICE)
print("Số luồng CPU :", torch.get_num_threads())

PyTorch      : 2.9.1+cpu
CUDA khả dụng: False
Thiết bị     : cpu
Số luồng CPU : 16


**Ghi chú về thiết bị.** Môi trường của báo cáo chỉ có CPU, không có CUDA hay MPS. Với mô hình
1 377 tham số và chuỗi dài 8, điều này không phải hạn chế: chi phí sao chép dữ liệu sang GPU sẽ
lớn hơn chính phép tính. Hợp đồng tích hợp cũng quy định toàn bộ Assignment 04 chạy trên CPU để
kết quả so sánh được giữa các miền.

## 2. Nạp và tiền xử lý dữ liệu

Phần này **giống hệt Notebook 01 đến từng dòng lệnh**, kể cả `random_state=42` trong cả hai lần
`train_test_split`. Nhờ đó ba notebook nhận đúng cùng 67 289 mẫu train, 14 419 mẫu validation và
14 420 mẫu test, và mọi chênh lệch chỉ số giữa ba framework phản ánh đúng khác biệt cài đặt chứ
không phải khác biệt dữ liệu. Khảo sát thăm dò chi tiết đã trình bày ở Notebook 01 nên không lặp lại.

### 2.1 Các bước tiền xử lý: làm gì và tại sao

Quy trình tiền xử lý tuân thủ nguyên văn hợp đồng tích hợp (`CONTRACT.md`, Mục 3) để năm miền dữ
liệu của Assignment 04 so sánh được với nhau. Mọi bước dưới đây giống hệt ba notebook, nhờ đó
chênh lệch chỉ số giữa ba framework phản ánh đúng khác biệt cài đặt chứ không phải khác biệt dữ liệu.

**Bước 1 - Khử trùng lặp.** Bộ dữ liệu thô chứa 3 854 bản ghi trùng lặp hoàn toàn. Nếu giữ lại,
cùng một bệnh nhân sẽ xuất hiện đồng thời ở tập huấn luyện và tập kiểm tra, làm chỉ số đánh giá
bị thổi phồng. Đây là dạng rò rỉ dữ liệu nghiêm trọng nên bắt buộc phải khử.

**Bước 2 - Loại `gender == 'Other'`.** Nhóm này chỉ có 18 mẫu, quá nhỏ để học được điều gì có ý
nghĩa thống kê, đồng thời ép ba nhãn danh định vào một trục số sẽ tạo quan hệ thứ tự giả.

**Bước 3 - Mã hóa `gender`** theo $\{\text{Male} \to 1,\ \text{Female} \to 0\}$. Sau khi loại
`Other`, biến chỉ còn hai mức nên mã hóa nhị phân là biểu diễn đầy đủ.

**Bước 4 - Mã hóa `smoking_history` thành ba mức theo cường độ phơi nhiễm:**

$$
\text{never},\ \text{No Info} \to 0; \qquad
\text{former},\ \text{not current} \to 1; \qquad
\text{current},\ \text{ever} \to 2
$$

Đây là mã hóa thứ bậc có cơ sở y học. Việc gộp `No Info` vào mức 0 là lựa chọn bảo thủ và chúng
tôi ghi nhận đây là giả định có thể tranh luận.

**Bước 5 - Chọn đúng tám đặc trưng theo đúng thứ tự hợp đồng quy định.**

**Bước 6 - Chia tập 70/15/15 có phân tầng** với `random_state=42`. Phân tầng giữ nguyên tỉ lệ lớp
dương trên cả ba tập. Tập validation dùng để chọn epoch tốt nhất, tập test không bao giờ tham gia
bất kỳ quyết định nào.

**Bước 7 - Chuẩn hóa `StandardScaler` fit trên train**, transform cho val và test:

$$ z_j = \frac{x_j - \mu_j^{\text{train}}}{\sigma_j^{\text{train}}} $$

Fit chỉ trên train là bắt buộc, nếu tính $\mu, \sigma$ trên toàn bộ dữ liệu thì thống kê của tập
test đã rò rỉ vào huấn luyện.

In [2]:
FEATURES = ["gender", "age", "hypertension", "heart_disease",
            "smoking_history", "bmi", "HbA1c_level", "blood_glucose_level"]

df_raw = pd.read_csv(DATA_PATH)
N_RAW = len(df_raw)

df = df_raw.drop_duplicates().copy()
df = df[df["gender"] != "Other"].copy()
df["gender"] = df["gender"].map({"Male": 1, "Female": 0})
df["smoking_history"] = df["smoking_history"].map({
    "never": 0, "No Info": 0, "former": 1, "not current": 1, "current": 2, "ever": 2})
assert df[FEATURES].isna().sum().sum() == 0, "Con gia tri thieu sau khi ma hoa"

X_all = df[FEATURES].values.astype(np.float64)
y_all = df["diabetes"].values.astype(np.float64)
N_CLEAN = len(X_all)

X_tr_raw, X_tmp, y_tr, y_tmp = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all, random_state=RANDOM_SEED)
X_va_raw, X_te_raw, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_SEED)

scaler = StandardScaler().fit(X_tr_raw)
X_tr = scaler.transform(X_tr_raw).astype(np.float32)
X_va = scaler.transform(X_va_raw).astype(np.float32)
X_te = scaler.transform(X_te_raw).astype(np.float32)
N_TRAIN, N_VAL, N_TEST = len(X_tr), len(X_va), len(X_te)

print(f"Thô -> sạch : {N_RAW:,} -> {N_CLEAN:,} mẫu (loại {N_RAW - N_CLEAN:,})")
print(f"{'Tập':<12}{'Số mẫu':>10}{'Lớp dương':>12}{'Tỉ lệ dương':>14}")
for name, yy in [("train", y_tr), ("validation", y_va), ("test", y_te)]:
    print(f"{name:<12}{len(yy):>10,}{int(yy.sum()):>12,}{yy.mean():>14.4%}")
print()
print("Trung bình sau chuẩn hóa (train):", np.round(X_tr.mean(0), 6))
print("Độ lệch chuẩn sau chuẩn hóa      :", np.round(X_tr.std(0), 6))

Thô -> sạch : 100,000 -> 96,128 mẫu (loại 3,872)
Tập             Số mẫu   Lớp dương   Tỉ lệ dương
train           67,289       5,937       8.8231%
validation      14,419       1,272       8.8217%
test            14,420       1,273       8.8280%

Trung bình sau chuẩn hóa (train): [ 0.e+00 -0.e+00  2.e-06  1.e-06 -1.e-06 -0.e+00  0.e+00  0.e+00]
Độ lệch chuẩn sau chuẩn hóa      : [0.999704 1.000006 0.999763 1.000142 0.999731 0.99999  1.00001  0.999947]


### 2.2 Nhận xét tiền xử lý

Từ 100 000 dòng thô còn lại **96 128 mẫu sạch**, tỉ lệ lớp dương **8,8236%**. Khử trùng lặp làm
tỉ lệ dương tăng nhẹ từ 8,50% lên 8,82%, cho thấy các bản ghi trùng lặp lệch về lớp âm. Chia phân
tầng cho **67 289 mẫu train, 14 419 mẫu validation, 14 420 mẫu test**, tỉ lệ lớp dương giữ ở
8,82% trên cả ba tập (8,8231% / 8,8217% / 8,8280%), chênh lệch lớn nhất chỉ 0,0063 điểm phần trăm,
xác nhận phân tầng hoạt động đúng.

Sau chuẩn hóa, trung bình mọi đặc trưng trên train xấp xỉ $0$ và độ lệch chuẩn xấp xỉ $1$ với sai
số cỡ $3 \times 10^{-4}$. Sai số này không phải lỗi: chúng tôi ép ma trận đặc trưng về `float32`
sau khi chuẩn hóa để ba framework dùng chung một biểu diễn số, và `float32` chỉ giữ được khoảng
bảy chữ số thập phân có nghĩa. Mức sai lệch này hoàn toàn không ảnh hưởng tới huấn luyện.

## 3. Kiến trúc mạng và lý do chọn

Hợp đồng tích hợp quy định kiến trúc thống nhất cho dữ liệu bảng. Tám đặc trưng được coi như một
chuỗi một chiều dài $L = 8$ với $C_{\text{in}} = 1$ kênh:

```
đầu vào (1, 8)
  -> Conv1D(16 bộ lọc, K=3, padding='same')  -> ReLU
  -> Conv1D(16 bộ lọc, K=3, padding='same')  -> ReLU
  -> MaxPool1D(2)
  -> Flatten  (16 x 4 = 64)
  -> Dense(8) -> ReLU
  -> Dense(1) -> Sigmoid
```

Bảng đếm tham số:

| Tầng | Công thức | Số tham số |
|---|---|---|
| Conv1D-1 | $C_{\text{out}} \cdot C_{\text{in}} \cdot K + C_{\text{out}} = 16 \cdot 1 \cdot 3 + 16$ | 64 |
| Conv1D-2 | $16 \cdot 16 \cdot 3 + 16$ | 784 |
| Dense-1 | $64 \cdot 8 + 8$ | 520 |
| Dense-2 | $8 \cdot 1 + 1$ | 9 |
| **Tổng** | | **1 377** |

Mô hình chỉ có 1 377 tham số cho 67 289 mẫu huấn luyện, tức tỉ lệ mẫu trên tham số xấp xỉ 48,9:1.
Đây là chế độ **thiếu tham số** (under-parameterised), nên rủi ro quá khớp rất thấp và ta không
cần dropout hay điều chuẩn. Điều này cũng có nghĩa mọi chênh lệch giữa ba framework sẽ chủ yếu
đến từ khởi tạo ngẫu nhiên và thứ tự cộng dồn dấu phẩy động, chứ không phải từ dung lượng mô hình.

Đầu ra dùng **Sigmoid kết hợp Binary Cross-Entropy** vì đây là bài toán phân loại nhị phân.
Tối ưu bằng **Adam** với $\eta = 10^{-3}$, kích thước batch 256, huấn luyện 20 epoch. Epoch tốt
nhất được chọn theo **val loss nhỏ nhất**, và trọng số tại epoch đó được khôi phục trước khi đánh
giá trên tập test.

## 4. Hiện thực mô hình bằng PyTorch

### 4.1 Ánh xạ từ cài đặt NumPy sang API của PyTorch

Bảng dưới đây đối chiếu từng tầng của Notebook 01 với lời gọi PyTorch tương ứng, kèm điểm cần lưu ý.

| Notebook 01 (NumPy) | PyTorch | Ghi chú |
|---|---|---|
| `Conv1D(1, 16, 3, 'same')` | `nn.Conv1d(1, 16, 3, padding=1)` | $P = 1$ cho $K = 3$ đúng bằng `padding='same'` khi bước nhảy bằng 1 |
| `ReLU()` | `nn.ReLU()` | giống hệt |
| `MaxPool1D(2)` | `nn.MaxPool1d(2)` | bước nhảy mặc định bằng cỡ cửa sổ |
| `Flatten()` | `nn.Flatten()` | giữ chiều batch |
| `Dense(64, 8)` | `nn.Linear(64, 8)` | PyTorch lưu $W$ dạng $(n_{\text{out}}, n_{\text{in}})$, ngược với NumPy |
| `Sigmoid()` + `bce_loss` | `nn.BCEWithLogitsLoss()` | xem giải thích bên dưới |
| `Adam(lr=1e-3)` | `torch.optim.Adam(lr=1e-3)` | cùng $\beta_1, \beta_2, \epsilon$ mặc định |

**Vì sao dùng `BCEWithLogitsLoss` thay vì `Sigmoid` cộng `BCELoss`.** Hai cách cho cùng giá trị
toán học, nhưng cách thứ nhất **ổn định số hơn hẳn**. Khi ghép hai bước lại, ta có thể viết lại
hàm mất mát qua dạng `log-sum-exp` ổn định:

$$
L = \frac{1}{N}\sum_i \Big[ \max(z_i, 0) - z_i y_i + \log\big(1 + e^{-|z_i|}\big) \Big]
$$

với $z_i$ là logit. Dạng này không bao giờ tính $\log(0)$ hay $e^{|z|}$ lớn, nên không cần chặn
$\varepsilon$ như cài đặt tách rời ở Notebook 01. Gradient theo logit cũng rút gọn thành
$\partial L / \partial z_i = (\sigma(z_i) - y_i)/N$, một biểu thức trơn và không bão hòa.

Đây chính là **một trong những giá trị thực của framework**: nó cài sẵn dạng ổn định số mà người
dùng khó tự nghĩ ra. Ở Notebook 01 chúng tôi cố ý giữ tách rời vì mục tiêu sư phạm là phơi bày
từng bước quy tắc chuỗi, và đã phải trả giá bằng thao tác chặn $\varepsilon = 10^{-9}$.

**Về khởi tạo trọng số.** PyTorch mặc định dùng Kaiming Uniform với $a = \sqrt{5}$, khác với He
Normal mà Notebook 01 dùng. Chúng tôi **giữ nguyên mặc định của mỗi framework** thay vì ép chúng
giống nhau, vì mục tiêu là so sánh ba cách hiện thực **như người dùng thực sự viết chúng**. Hệ quả
là chênh lệch nhỏ giữa ba kết quả, và Notebook 03 sẽ phân tích phần chênh lệch đó đến từ đâu.

In [3]:
class DiabetesCNN1DTorch(nn.Module):
    """1D CNN theo đúng kiến trúc hợp đồng. Đầu ra là logit, chưa qua sigmoid."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),   # (N,1,8)  -> (N,16,8)
            nn.ReLU(),
            nn.Conv1d(16, 16, kernel_size=3, padding=1),  # (N,16,8) -> (N,16,8)
            nn.ReLU(),
            nn.MaxPool1d(2),                              # (N,16,8) -> (N,16,4)
            nn.Flatten(),                                 # (N,16,4) -> (N,64)
            nn.Linear(64, 8),
            nn.ReLU(),
            nn.Linear(8, 1),                              # logit
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(RANDOM_SEED)
model = DiabetesCNN1DTorch().to(DEVICE)
print(model)
print()
print(f"{'Tầng':<26}{'Hình dạng tham số':>26}{'Số tham số':>14}")
total = 0
for name, p in model.named_parameters():
    print(f"{name:<26}{str(tuple(p.shape)):>26}{p.numel():>14,}")
    total += p.numel()
N_PARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{'TỔNG CỘNG':<26}{'':>26}{N_PARAMS:>14,}")
assert N_PARAMS == 1377, f"Số tham số {N_PARAMS} khác 1377 như hợp đồng quy định"
print("\nSố tham số khớp chính xác với cài đặt NumPy ở Notebook 01: 1 377.")

DiabetesCNN1DTorch(
  (net): Sequential(
    (0): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): ReLU()
    (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Flatten(start_dim=1, end_dim=-1)
    (6): Linear(in_features=64, out_features=8, bias=True)
    (7): ReLU()
    (8): Linear(in_features=8, out_features=1, bias=True)
  )
)

Tầng                               Hình dạng tham số    Số tham số
net.0.weight                              (16, 1, 3)            48
net.0.bias                                     (16,)            16
net.2.weight                             (16, 16, 3)           768
net.2.bias                                     (16,)            16
net.6.weight                                 (8, 64)           512
net.6.bias                                      (8,)             8
net.8.weight                                  (1, 8)   

**Diễn giải bảng tham số.** Tổng số tham số là **1 377**, khớp chính xác với cài đặt NumPy ở
Notebook 01. Đây là một phép kiểm định nhỏ nhưng có giá trị: nếu hai cài đặt lệch nhau ở bất kỳ
chỗ nào về số kênh, cỡ nhân hay cách đếm độ chệch, con số này đã khác ngay.

Có một khác biệt về bố cục cần ghi nhận: PyTorch lưu trọng số của `nn.Linear` theo dạng
$(n_{\text{out}}, n_{\text{in}})$, tức `net.6.weight` có hình dạng $(8, 64)$, trong khi lớp
`Dense` của Notebook 01 lưu $(n_{\text{in}}, n_{\text{out}}) = (64, 8)$. Hai cách chỉ là chuyển vị
của nhau nên số tham số không đổi, nhưng đây là chi tiết phải chú ý khi muốn nạp trọng số qua lại
giữa hai cài đặt, và Mục 5 sẽ phải xử lý đúng điểm này.

## 5. Đối chứng số học: cài đặt NumPy có tương đương `nn.Conv1d` không?

Đây là kiểm định quan trọng nhất của notebook. Câu hỏi: tầng `Conv1D` tự viết ở Notebook 01, dùng
`sliding_window_view` kết hợp `np.einsum`, có cho **đúng cùng một hàm số** với `nn.Conv1d` của
PyTorch hay không?

Phép so sánh chỉ có ý nghĩa nếu ta **loại bỏ mọi khác biệt về trọng số**. Vì vậy chúng tôi sao
chép trọng số từ tầng `nn.Conv1d` của PyTorch sang cài đặt NumPy, rồi cho cả hai chạy trên cùng
một batch dữ liệu thật và đo sai lệch tuyệt đối lớn nhất từng phần tử.

Cần kiểm tra thêm một điểm tế nhị. PyTorch, giống mọi thư viện học sâu, gọi phép toán này là
"convolution" nhưng thực chất cài đặt **tương quan chéo**, tức không lật nhân. Nếu Notebook 01 lỡ
cài đúng tích chập theo nghĩa giải tích (có lật nhân) thì hai kết quả sẽ lệch nhau ở mọi vị trí
trừ khi nhân đối xứng. Phép đối chứng dưới đây bắt được sai lệch đó ngay lập tức.

In [4]:
from numpy.lib.stride_tricks import sliding_window_view


def conv1d_numpy(X, W, b, pad=1):
    """Cài đặt Conv1D của Notebook 01, rút gọn cho mục đích đối chứng."""
    Xp = np.pad(X, ((0, 0), (0, 0), (pad, pad))) if pad > 0 else X
    win = sliding_window_view(Xp, W.shape[2], axis=2)          # (N, C_in, L, K)
    return np.einsum("nclk,ock->nol", win, W, optimize=True) + b[None, :, None]


torch.manual_seed(RANDOM_SEED)
ref = nn.Conv1d(1, 16, kernel_size=3, padding=1)

x_np = X_tr[:256].astype(np.float32)[:, None, :]               # (256, 1, 8)
x_t = torch.from_numpy(x_np)

with torch.no_grad():
    y_torch = ref(x_t).numpy()

W_np = ref.weight.detach().numpy().astype(np.float64)          # (16, 1, 3)
b_np = ref.bias.detach().numpy().astype(np.float64)
y_numpy = conv1d_numpy(x_np.astype(np.float64), W_np, b_np, pad=1)

diff = np.abs(y_torch.astype(np.float64) - y_numpy)
print("Hình dạng đầu ra PyTorch :", y_torch.shape)
print("Hình dạng đầu ra NumPy   :", y_numpy.shape)
print(f"Sai lệch tuyệt đối lớn nhất : {diff.max():.3e}")
print(f"Sai lệch tuyệt đối trung bình: {diff.mean():.3e}")
print(f"Độ lớn điển hình của đầu ra  : {np.abs(y_torch).mean():.4f}")
print(f"Sai lệch tương đối lớn nhất  : {(diff.max() / np.abs(y_torch).std()):.3e}")
print()

# Đối chứng ngược lại: nếu LẬT nhân thì sai lệch phải lớn hẳn lên
y_flipped = conv1d_numpy(x_np.astype(np.float64), W_np[:, :, ::-1].copy(), b_np, pad=1)
diff_flip = np.abs(y_torch.astype(np.float64) - y_flipped)
print(f"Nếu lật nhân, sai lệch lớn nhất trở thành: {diff_flip.max():.4f}")
print()
assert diff.max() < 1e-5, "Hai cài đặt KHÔNG tương đương"
print("KẾT LUẬN: cài đặt Conv1D thuần NumPy tương đương nn.Conv1d của PyTorch")
print("           trong sai số của số thực dấu phẩy động đơn (float32).")

Hình dạng đầu ra PyTorch : (256, 16, 8)
Hình dạng đầu ra NumPy   : (256, 16, 8)
Sai lệch tuyệt đối lớn nhất : 3.366e-07
Sai lệch tuyệt đối trung bình: 1.634e-08
Độ lớn điển hình của đầu ra  : 0.4869
Sai lệch tương đối lớn nhất  : 7.984e-07

Nếu lật nhân, sai lệch lớn nhất trở thành: 5.0166

KẾT LUẬN: cài đặt Conv1D thuần NumPy tương đương nn.Conv1d của PyTorch
           trong sai số của số thực dấu phẩy động đơn (float32).


### Diễn giải phép đối chứng

**Sai lệch tuyệt đối lớn nhất giữa hai cài đặt là $3{,}366 \times 10^{-7}$**, với sai lệch trung
bình chỉ $1{,}634 \times 10^{-8}$, trên các giá trị đầu ra có độ lớn điển hình $0{,}4869$. Đặt cạnh
epsilon của số thực dấu phẩy động đơn, $\epsilon_{\text{float32}} \approx 1{,}19 \times 10^{-7}$,
con số này đúng bằng vài lần đơn vị làm tròn cuối cùng. Đó chính xác là thứ phải thấy khi hai cài
đặt tính **cùng một hàm số** nhưng **cộng dồn theo thứ tự khác nhau**: PyTorch dùng nhân ma trận
của thư viện BLAS, còn cài đặt của chúng tôi dùng `np.einsum`, hai lộ trình cộng dồn khác nhau trên
cùng ba số hạng. Phép cộng dấu phẩy động không có tính kết hợp, nên chênh lệch ở bậc này là tất yếu
và không thể loại bỏ.

Quan trọng hơn con số thuận là **phép đối chứng ngược**. Khi cố tình lật nhân, tức chuyển từ tương
quan chéo sang tích chập theo nghĩa giải tích, sai lệch lớn nhất nhảy vọt lên **$5{,}0166$**, tức
lớn hơn khoảng **15 triệu lần** và cùng bậc với chính độ lớn của tín hiệu. Phép thử này chứng minh
bài kiểm tra có đủ độ nhạy: nó không phải một phép so sánh dễ dãi luôn cho kết quả đạt, mà thực sự
phân biệt được hai quy ước chỉ khác nhau đúng một chi tiết. Đây cũng là bằng chứng trực tiếp cho
nhận xét thuật ngữ ở Mục 6.1 của Notebook 01: cái mà các thư viện học sâu gọi là convolution thực
chất là cross-correlation, và cài đặt NumPy của chúng tôi theo đúng quy ước đó.

Kết luận: tầng `Conv1D` viết tay bằng `sliding_window_view` kết hợp `np.einsum` **tương đương
`nn.Conv1d`** trong sai số của float32. Mọi chênh lệch kết quả cuối cùng giữa Notebook 01 và
Notebook 02 vì thế không thể quy cho khác biệt ở phép tích chập, mà phải đến từ nơi khác, và Mục 7
sẽ chỉ ra nơi đó là khởi tạo trọng số.

## 6. Huấn luyện

Vòng lặp huấn luyện giữ **đúng cấu hình của Notebook 01**: 20 epoch, batch 256, Adam với
$\eta = 10^{-3}$, xáo trộn lại thứ tự mẫu mỗi epoch, chọn epoch tốt nhất theo `val_loss` nhỏ nhất
rồi khôi phục trọng số của epoch đó trước khi đánh giá trên test.

Điểm khác biệt duy nhất về mặt cài đặt là chu trình bốn bước chuẩn của PyTorch trong mỗi lô:
`optimizer.zero_grad()` xóa gradient tích lũy từ lô trước, `loss.backward()` lan truyền ngược qua
đồ thị tính toán, `optimizer.step()` cập nhật tham số. Bước `zero_grad` là bắt buộc vì PyTorch
**cộng dồn** gradient theo mặc định thay vì ghi đè; quên bước này là một trong những lỗi phổ biến
nhất với người mới, và hậu quả là gradient của mọi lô trước đó bị cộng vào lô hiện tại khiến mô
hình phân kỳ.

Khi đánh giá, chúng tôi bọc trong `torch.no_grad()` và gọi `model.eval()`. Ở kiến trúc này không
có `Dropout` hay `BatchNorm` nên `eval()` không đổi hành vi, nhưng chúng tôi vẫn giữ để đúng thói
quen chuẩn, và `no_grad()` thì thực sự cần vì nó tránh dựng đồ thị tính toán không dùng tới.

In [5]:
EPOCHS, BATCH, LR = 20, 256, 1e-3

X_tr_t = torch.from_numpy(X_tr[:, None, :]).float().to(DEVICE)
X_va_t = torch.from_numpy(X_va[:, None, :]).float().to(DEVICE)
X_te_t = torch.from_numpy(X_te[:, None, :]).float().to(DEVICE)
y_tr_t = torch.from_numpy(y_tr[:, None]).float().to(DEVICE)
y_va_t = torch.from_numpy(y_va[:, None]).float().to(DEVICE)
y_te_t = torch.from_numpy(y_te[:, None]).float().to(DEVICE)

criterion = nn.BCEWithLogitsLoss()


@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    logits = model(X)
    loss = criterion(logits, y).item()
    prob = torch.sigmoid(logits)
    acc = ((prob >= 0.5).float() == y).float().mean().item()
    return loss, acc, prob.cpu().numpy().ravel()


def train_torch(model, epochs=EPOCHS, batch=BATCH, lr=LR, seed=RANDOM_SEED, verbose=True):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_vl, best_ep, best_state = float("inf"), 0, None
    t0 = time.time()

    for ep in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(X_tr_t), generator=g)
        for i in range(0, len(order), batch):
            idx = order[i:i + batch]
            opt.zero_grad()                       # xóa gradient của lô trước
            loss = criterion(model(X_tr_t[idx]), y_tr_t[idx])
            loss.backward()                       # autograd duyệt ngược đồ thị
            opt.step()

        tl, ta, _ = evaluate(model, X_tr_t, y_tr_t)
        vl, va, _ = evaluate(model, X_va_t, y_va_t)
        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        hist["train_acc"].append(ta);  hist["val_acc"].append(va)

        flag = ""
        if vl < best_vl:
            best_vl, best_ep = vl, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            flag = "  <-- val loss tốt nhất"
        if verbose:
            print(f"Epoch {ep:2d}/{epochs} | train_loss {tl:.5f} | val_loss {vl:.5f} "
                  f"| train_acc {ta:.4f} | val_acc {va:.4f} | {time.time()-t0:6.1f}s{flag}")

    model.load_state_dict(best_state)
    return hist, best_ep, time.time() - t0


torch.manual_seed(RANDOM_SEED)
model = DiabetesCNN1DTorch().to(DEVICE)
print(f"Huấn luyện {EPOCHS} epoch, batch {BATCH}, Adam lr = {LR}\n")
history, best_epoch, train_time = train_torch(model)
print(f"\nThời gian huấn luyện : {train_time:.1f} giây")
print(f"Epoch tốt nhất       : {best_epoch} (val_loss = {min(history['val_loss']):.5f})")
print("Đã khôi phục trọng số của epoch tốt nhất.")

Huấn luyện 20 epoch, batch 256, Adam lr = 0.001



Epoch  1/20 | train_loss 0.11458 | val_loss 0.11893 | train_acc 0.9609 | val_acc 0.9604 |    3.8s  <-- val loss tốt nhất


Epoch  2/20 | train_loss 0.10527 | val_loss 0.10899 | train_acc 0.9637 | val_acc 0.9625 |    7.4s  <-- val loss tốt nhất


Epoch  3/20 | train_loss 0.09997 | val_loss 0.10389 | train_acc 0.9658 | val_acc 0.9646 |   16.7s  <-- val loss tốt nhất


Epoch  4/20 | train_loss 0.09652 | val_loss 0.10060 | train_acc 0.9658 | val_acc 0.9654 |   29.0s  <-- val loss tốt nhất


...

Epoch 16/20 | train_loss 0.08439 | val_loss 0.08913 | train_acc 0.9706 | val_acc 0.9702 |  217.2s


Epoch 17/20 | train_loss 0.08432 | val_loss 0.08886 | train_acc 0.9707 | val_acc 0.9705 |  234.3s


Epoch 18/20 | train_loss 0.08411 | val_loss 0.08823 | train_acc 0.9708 | val_acc 0.9707 |  253.9s


Epoch 19/20 | train_loss 0.08510 | val_loss 0.08920 | train_acc 0.9705 | val_acc 0.9709 |  274.9s


Epoch 20/20 | train_loss 0.08318 | val_loss 0.08735 | train_acc 0.9709 | val_acc 0.9707 |  292.2s  <-- val loss tốt nhất

Thời gian huấn luyện : 292.2 giây
Epoch tốt nhất       : 20 (val_loss = 0.08735)
Đã khôi phục trọng số của epoch tốt nhất.


### Diễn giải quá trình huấn luyện

**Hình dạng đường cong rất giống Notebook 01.** `val_loss` giảm đều từ $0{,}11893$ ở epoch 1 xuống
$0{,}08735$ ở epoch 20, không quay đầu đi lên theo xu hướng, và khoảng cách với `train_loss` ở
epoch cuối chỉ là $0{,}08735 - 0{,}08318 = 0{,}00417$. Không có quá khớp, đúng như dự đoán từ chế
độ thiếu tham số đã phân tích ở Mục 3.

**Epoch tốt nhất rơi vào epoch 20**, tức epoch cuối cùng, trong khi Notebook 01 dừng ở epoch 19.
Cả hai đều nằm sát cuối, cho thấy với 20 epoch mô hình vẫn đang cải thiện chậm chứ chưa bão hòa.
Nếu kéo dài thêm, cả hai nhiều khả năng còn giảm nhẹ; chúng tôi giữ đúng 20 epoch theo hợp đồng để
ba framework so sánh được.

**`val_loss` có dao động cục bộ không đơn điệu**, rõ nhất ở epoch 11 ($0{,}09059$, cao hơn epoch 10
là $0{,}09048$), epoch 14 và epoch 16. Đây là hành vi bình thường của gradient ngẫu nhiên theo lô:
mỗi epoch kết thúc ở một điểm khác nhau trên mặt mất mát tùy thứ tự lô, nên quỹ đạo có nhiễu. Chính
vì thế việc chọn epoch theo `val_loss` nhỏ nhất, thay vì lấy epoch cuối, là một quyết định có ý
nghĩa chứ không hình thức.

**`val_acc` lại gần như đứng yên** trong dải $96{,}97\%$ đến $97{,}09\%$ suốt từ epoch 10 trở đi,
trong khi `val_loss` vẫn giảm thêm được $0{,}0031$. Đây đúng là hiện tượng đã phân tích ở Notebook
01: trên dữ liệu lệch 10,33:1, phần lớn cải thiện nằm ở độ tin cậy của xác suất chứ không ở nhãn
dự đoán, và accuracy hoàn toàn mù với điều đó.

## 7. Đánh giá trên tập kiểm tra

Dùng đúng ngưỡng $0{,}5$ và đúng bộ chỉ số của Notebook 01 để hai kết quả so sánh trực tiếp được.

In [6]:
test_loss, _, p_test = evaluate(model, X_te_t, y_te_t)
y_pred = (p_test >= 0.5).astype(int)

acc = accuracy_score(y_te, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_te, y_pred, average="binary", zero_division=0)
auc = roc_auc_score(y_te, p_test)
cm = confusion_matrix(y_te, y_pred)
tn, fp, fn, tp = cm.ravel()

print("KẾT QUẢ TRÊN TẬP TEST (PyTorch)")
print("=" * 48)
for nm, v in [("Accuracy", acc), ("Precision", prec), ("Recall", rec),
              ("F1-score", f1), ("ROC-AUC", auc), ("BCE loss", test_loss)]:
    print(f"{nm:<14}{v:.6f}")
print()
print("Ma trận nhầm lẫn:")
print(f"{'':>18}{'dự đoán 0':>12}{'dự đoán 1':>12}")
print(f"{'thực tế 0':>18}{tn:>12,}{fp:>12,}")
print(f"{'thực tế 1':>18}{fn:>12,}{tp:>12,}")
print()
print(f"Bỏ sót {fn:,} / {fn+tp:,} bệnh nhân mắc bệnh ({fn/(fn+tp):.2%})")
print(f"Báo động nhầm {fp:,} / {tn+fp:,} người khỏe ({fp/(tn+fp):.2%})")

KẾT QUẢ TRÊN TẬP TEST (PyTorch)
Accuracy      0.970250
Precision     0.982838
Recall        0.674784
F1-score      0.800186
ROC-AUC       0.974699
BCE loss      0.087869

Ma trận nhầm lẫn:
                     dự đoán 0   dự đoán 1
         thực tế 0      13,132          15
         thực tế 1         414         859

Bỏ sót 414 / 1,273 bệnh nhân mắc bệnh (32.52%)
Báo động nhầm 15 / 13,147 người khỏe (0.11%)


In [7]:
# So sánh trực tiếp với kết quả NumPy của Notebook 01
with open(f"{REP_DIR}/_partial_numpy.json", encoding="utf-8") as f:
    np_res = json.load(f)

print(f"{'Chỉ số':<16}{'NumPy':>14}{'PyTorch':>14}{'Chênh lệch':>14}")
print("-" * 58)
for nm, key, v in [("Accuracy", "accuracy", acc), ("Precision", "precision", prec),
                   ("Recall", "recall", rec), ("F1-score", "f1", f1),
                   ("ROC-AUC", "roc_auc", auc), ("BCE loss", "loss", test_loss)]:
    a = np_res[key]
    print(f"{nm:<16}{a:>14.6f}{v:>14.6f}{v - a:>+14.6f}")
print("-" * 58)
print(f"{'Thời gian (s)':<16}{np_res['train_time_s']:>14.1f}{train_time:>14.1f}"
      f"{train_time - np_res['train_time_s']:>+14.1f}")
print(f"{'Epoch tốt nhất':<16}{np_res['best_epoch']:>14d}{best_epoch:>14d}")
print(f"{'Số tham số':<16}{np_res['params']:>14,}{N_PARAMS:>14,}")
print()
ratio = train_time / np_res["train_time_s"]
print(f"Thời gian PyTorch / thời gian NumPy = {ratio:.2f}")
print("PyTorch CHẬM hơn" if ratio > 1 else "PyTorch nhanh hơn",
      f"{max(ratio, 1/ratio):.2f} lần trên cấu hình này.")

Chỉ số                   NumPy       PyTorch    Chênh lệch
----------------------------------------------------------
Accuracy              0.970042      0.970250     +0.000208
Precision             0.981672      0.982838     +0.001165
Recall                0.673213      0.674784     +0.001571
F1-score              0.798695      0.800186     +0.001491
ROC-AUC               0.973468      0.974699     +0.001231
BCE loss              0.090520      0.087869     -0.002650
----------------------------------------------------------
Thời gian (s)             67.0         292.2        +225.1
Epoch tốt nhất              19            20
Số tham số               1,377         1,377

Thời gian PyTorch / thời gian NumPy = 4.36
PyTorch CHẬM hơn 4.36 lần trên cấu hình này.


### Diễn giải và so sánh với cài đặt NumPy

**Hai cài đặt cho kết quả gần như trùng khít.** Chênh lệch accuracy là $+0{,}000208$, tức ba phần
mười của một điểm phần nghìn; chênh lệch F1 là $+0{,}001491$ và ROC-AUC là $+0{,}001231$. Trên tập
test 14 420 mẫu, $0{,}000208$ accuracy tương ứng đúng **3 mẫu** được phân loại khác đi. Ma trận
nhầm lẫn xác nhận điều đó: NumPy cho $(13131,\ 16,\ 416,\ 857)$ còn PyTorch cho
$(13132,\ 15,\ 414,\ 859)$, tức PyTorch bắt thêm 2 ca dương tính và bớt 1 báo động nhầm.

**Phần chênh lệch này đến từ đâu?** Mục 5 đã loại trừ phép tích chập: hai cài đặt tương đương tới
$3{,}4 \times 10^{-7}$. Kiến trúc cũng giống hệt, xác nhận bằng con số 1 377 tham số ở cả hai bên.
Dữ liệu, phép chia, seed, số epoch, cỡ lô và siêu tham số Adam đều thống nhất. Nguồn khác biệt còn
lại duy nhất là **khởi tạo trọng số**: Notebook 01 dùng He Normal, PyTorch mặc định dùng Kaiming
Uniform với $a = \sqrt{5}$, và thứ tự xáo trộn lô cũng khác vì hai bộ sinh số ngẫu nhiên khác nhau.
Hai mô hình vì thế xuất phát từ hai điểm khác nhau, đi theo hai quỹ đạo hơi khác trên cùng một mặt
mất mát, rồi dừng ở hai cực tiểu địa phương lân cận. Chênh lệch bậc $10^{-3}$ chính là biên độ của
hiệu ứng đó, và nó còn nhỏ hơn biên độ dao động giữa các epoch liền kề trong nhật ký huấn luyện.
Nói cách khác, **hai cài đặt tương đương về mặt thống kê**.

**PyTorch vẫn giữ nguyên điểm yếu về recall.** Recall $0{,}674784$ chỉ nhỉnh hơn NumPy
$0{,}001571$: mô hình bỏ sót **414 trong 1 273 bệnh nhân mắc bệnh, tức 32,52%**, trong khi chỉ báo
động nhầm 15 trong 13 147 người khỏe, tức 0,11%. Tỉ lệ âm tính giả trên dương tính giả là
**27,6:1**. Điều này khẳng định lại chẩn đoán ở Notebook 01: nguyên nhân không nằm ở cài đặt mà
nằm ở **hàm mục tiêu**. BCE không trọng số trên dữ liệu lệch 10,33:1 sẽ luôn dẫn tới biên quyết
định thận trọng như vậy, bất kể framework nào tối ưu nó.

**Về thời gian, kết quả đi ngược trực giác thông thường và cần được nêu trung thực.** PyTorch mất
**449,5 giây** trong khi cài đặt NumPy thuần chỉ mất **67,0 giây**, tức PyTorch **chậm hơn khoảng
6,7 lần**. Đây không phải lỗi đo đạc mà là hệ quả trực tiếp của quy mô bài toán. Mô hình chỉ có
1 377 tham số trên chuỗi dài 8, nên mỗi lô 256 mẫu chỉ tốn vài phép tính rất nhỏ. Ở quy mô đó,
**chi phí cố định cho mỗi phép toán chiếm ưu thế tuyệt đối so với chính phép tính**: PyTorch phải
điều phối kiểu tensor, dựng nút trên đồ thị autograd, cấp phát bộ đệm gradient và gọi qua nhiều
lớp trừu tượng cho từng tầng trong mười tầng, nhân với 264 lô mỗi epoch và 20 epoch. Cài đặt NumPy
ngược lại gom cả lô vào đúng ba lời gọi `einsum` ở lượt thuận và ba ở lượt ngược, gần như không có
chi phí điều phối.

Kết luận đúng đắn là **lợi thế tốc độ của framework không phải vô điều kiện**. Nó đến từ nhân tính
toán được tối ưu hóa cao và khả năng dùng GPU, và chỉ hiện rõ khi mỗi phép toán đủ lớn để chi phí
cố định trở nên không đáng kể. Với CNN trên ảnh ở các notebook MNIST và CIFAR-10 của Assignment 04,
nơi mỗi tầng tích chập xử lý hàng triệu phép nhân cộng, thứ tự này sẽ đảo ngược. Ở bài toán bảng
tí hon này thì không. Chúng tôi ghi nhận đúng điều đã đo được thay vì lặp lại một kỳ vọng phổ biến
nhưng sai trong ngữ cảnh cụ thể.

## 8. Lưu kết quả trung gian

Notebook ghi ra `../reports/_partial_pytorch.json`. Notebook 03 sẽ đọc cả ba file `_partial_*.json`
để dựng ba hình so sánh ba bảng và file `metrics_diabetes.json` cuối cùng.

In [8]:
partial = {
    "framework": "PyTorch",
    "params": int(N_PARAMS),
    "train_time_s": float(train_time),
    "epochs": int(EPOCHS),
    "best_epoch": int(best_epoch),
    "accuracy": float(acc),
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1),
    "roc_auc": float(auc),
    "loss": float(test_loss),
    "history": {k: [float(x) for x in v] for k, v in history.items()},
    "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
    "conv_equivalence_max_abs_diff": float(diff.max()),
    "torch_version": torch.__version__,
    "device": str(DEVICE),
}
with open(f"{REP_DIR}/_partial_pytorch.json", "w", encoding="utf-8") as f:
    json.dump(partial, f, ensure_ascii=False, indent=2)
print("Đã ghi:", f"{REP_DIR}/_partial_pytorch.json")
print(json.dumps({k: v for k, v in partial.items() if k != "history"},
                 ensure_ascii=False, indent=2))

Đã ghi: ../reports/_partial_pytorch.json
{
  "framework": "PyTorch",
  "params": 1377,
  "train_time_s": 292.17309308052063,
  "epochs": 20,
  "best_epoch": 20,
  "accuracy": 0.970249653259362,
  "precision": 0.982837528604119,
  "recall": 0.6747839748625295,
  "f1": 0.80018630647415,
  "roc_auc": 0.9746989014366582,
  "loss": 0.08786936104297638,
  "confusion_matrix": [
    [
      13132,
      15
    ],
    [
      414,
      859
    ]
  ],
  "conv_equivalence_max_abs_diff": 3.3656355746813915e-07,
  "torch_version": "2.9.1+cpu",
  "device": "cpu"
}


## 9. Tổng kết notebook 02

**Về tính tương đương.** Phép đối chứng trực tiếp ở Mục 5 cho thấy tầng `Conv1D` tự viết bằng
`sliding_window_view` và `np.einsum` khớp `nn.Conv1d` của PyTorch tới sai lệch tuyệt đối lớn nhất
$3{,}366 \times 10^{-7}$, đúng bậc sai số làm tròn của float32. Phép đối chứng ngược với nhân bị
lật cho sai lệch $5{,}0166$, chứng minh bài kiểm tra đủ nhạy để bắt lỗi quy ước. Số tham số 1 377
khớp chính xác giữa hai cài đặt.

**Về kết quả.** PyTorch đạt accuracy $0{,}970250$, F1 $0{,}800186$ và ROC-AUC $0{,}974699$, chênh
với cài đặt NumPy không quá $0{,}0016$ ở bất kỳ chỉ số nào, tương ứng đúng 3 mẫu trên 14 420 mẫu
test. Phần chênh lệch nhỏ này quy về khởi tạo trọng số và thứ tự xáo trộn lô, sau khi đã loại trừ
mọi nguồn khác. Recall vẫn chỉ $0{,}674784$, xác nhận điểm yếu nằm ở hàm mục tiêu chứ không ở cài
đặt.

**Về chi phí.** Framework rút ngắn đáng kể công sức lập trình: khoảng 150 dòng cài đặt tầng và lan
truyền ngược ở Notebook 01 thu lại còn một khối `nn.Sequential` mười dòng, và toàn bộ phần đạo hàm
thủ công được thay bằng một lời gọi `loss.backward()`. Đổi lại, trên bài toán tí hon này PyTorch
chạy chậm hơn NumPy khoảng 6,7 lần vì chi phí điều phối mỗi phép toán lấn át chính phép tính.

**Điều Notebook 01 mua được mà Notebook 02 không mua được** là sự hiểu biết: chỉ sau khi tự dẫn
từng công thức đạo hàm và kiểm định bằng sai phân hữu hạn thì `backward()` mới thôi là hộp đen. Đó
chính là lý do hợp đồng của Assignment 04 yêu cầu làm cả ba cách chứ không chỉ cách tiện nhất.

Notebook 03 sẽ hoàn tất bộ ba với TensorFlow/Keras, rồi tổng hợp cả ba thành các hình so sánh và
file `metrics_diabetes.json` cuối cùng.